# Mastering Imperfect Information with Deep Recurrent Q-Networks
## Leduc Hold'em — DRQN vs DQN vs Heuristic vs Random

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/DLPW/blob/master/notebooks/DLPW_colab.ipynb)

**Google Colab Version**

This notebook demonstrates the modular DRQN training pipeline on Google Colab.

**Setup Options:**
1. **From GitHub** (recommended): Clone the repository
2. **From Google Drive**: Upload project folder to Drive
3. **Standalone**: All code embedded in notebook

**Runtime**: GPU recommended (Runtime → Change runtime type → GPU)

## 🔧 Setup & Installation

In [ ]:
# Install dependencies
!pip install -q rlcard torch matplotlib

import sys
import os
import torch

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Dependencies installed")
print(f"✓ Device: {device}")
print(f"✓ PyTorch version: {torch.__version__}")

### Choose Setup Method

**Option 1: Clone from GitHub** (recommended if repo is public)

In [ ]:
# Uncomment and run if using GitHub
# !git clone https://github.com/YOUR_USERNAME/DLPW.git
# %cd DLPW
# sys.path.insert(0, '/content/DLPW')

**Option 2: Mount Google Drive**

In [ ]:
# Uncomment to use Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# 
# # Update this path to your DLPW folder location in Drive
# project_path = '/content/drive/MyDrive/DLPW'
# %cd {project_path}
# sys.path.insert(0, project_path)

**Option 3: Standalone Mode** (all code in notebook)

Run the following cells to define all modules inline:

In [ ]:
# ============================================================
# CONFIG (inline version)
# ============================================================

import torch

# Environment
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ENV_NAME = 'leduc-holdem'

# Model Architecture
HIDDEN_SIZE = 64

# Training Hyperparameters
LEARNING_RATE = 1e-3
GAMMA = 0.99

# Exploration
EPSILON_START = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.9995

# Experience Replay
BUFFER_CAPACITY = 5000
BATCH_SIZE = 64
MIN_REPLAY_SIZE = 256
TARGET_UPDATE_FREQ = 50

# Training Schedule (reduced for Colab)
NUM_EPISODES_PHASE1 = 5000   # Reduced from 15000 for faster training
NUM_EPISODES_PHASE2 = 5000   # Reduced from 15000
EVALUATE_EVERY = 500         # Evaluate more frequently
EVALUATE_NUM = 500           # Fewer evaluation games
NUM_EVAL_HANDS = 1000        # Reduced final evaluation

# Paths
OUTPUT_DIR = '/content/outputs'
MODEL_DIR = f'{OUTPUT_DIR}/models'
PLOT_DIR = f'{OUTPUT_DIR}/plots'

# Action Mapping
ACTION_NAMES = {
    'call': 'Call',
    'raise': 'Raise',
    'fold': 'Fold',
    'check': 'Check'
}

# Create directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

print("✓ Configuration loaded")
print(f"  Phase 1: {NUM_EPISODES_PHASE1} episodes")
print(f"  Phase 2: {NUM_EPISODES_PHASE2} episodes")
print(f"  Device: {DEVICE}")

In [ ]:
# ============================================================
# MODELS (inline version)
# ============================================================

import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import random

class LeducDRQN(nn.Module):
    """LSTM-based Q-Network for sequential decision making."""
    
    def __init__(self, state_shape, num_actions, hidden_size=64):
        super().__init__()
        self.fc1 = nn.Linear(state_shape, hidden_size)
        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            batch_first=True
        )
        self.fc2 = nn.Linear(hidden_size, num_actions)

    def forward(self, x, hidden_state=None):
        batch, seq_len, feat = x.shape
        x_flat = x.reshape(batch * seq_len, feat)
        x_flat = F.relu(self.fc1(x_flat))
        x = x_flat.reshape(batch, seq_len, -1)
        lstm_out, hidden_state = self.lstm(x, hidden_state)
        last_step = lstm_out[:, -1, :]
        q_values = self.fc2(last_step)
        return q_values, hidden_state


class SequenceReplayBuffer:
    """Stores complete episode sequences."""
    
    def __init__(self, capacity=5000):
        self.buffer = deque(maxlen=capacity)

    def push(self, obs_sequence, action, reward):
        self.buffer.append((list(obs_sequence), action, reward))

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

print("✓ Models defined")

In [ ]:
# ============================================================
# AGENTS (inline version)
# ============================================================

import torch.optim as optim
import numpy as np
import copy

class DRQNAgent:
    """DRQN Agent with sequence replay and target network."""
    
    def __init__(self, state_shape, num_actions, device,
                 hidden_size=64, lr=1e-3, gamma=0.99,
                 epsilon_start=1.0, epsilon_min=0.05, epsilon_decay=0.9995,
                 buffer_capacity=5000, batch_size=64, min_replay=256,
                 target_update_freq=50):
        self.use_raw = False
        self.num_actions = num_actions
        self.device = device
        self.gamma = gamma
        self.batch_size = batch_size
        self.min_replay = min_replay
        self.target_update_freq = target_update_freq
        self._train_steps = 0

        self.model = LeducDRQN(state_shape, num_actions, hidden_size).to(device)
        self.target_model = copy.deepcopy(self.model).to(device)
        self.target_model.eval()
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)

        self.epsilon = epsilon_start
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay

        self.replay_buffer = SequenceReplayBuffer(buffer_capacity)
        self.current_hand_sequence = []

    def _seq_to_tensor(self, sequence):
        arr = np.array(sequence, dtype=np.float32)
        return torch.from_numpy(arr).unsqueeze(0).to(self.device)

    def _greedy_action(self, legal_actions):
        seq_t = self._seq_to_tensor(self.current_hand_sequence)
        with torch.no_grad():
            q_values, _ = self.model(seq_t)
            q_values = q_values.cpu().numpy()[0]
        masked = np.full(self.num_actions, -np.inf)
        for a in legal_actions:
            masked[a] = q_values[a]
        return int(np.argmax(masked))

    def step(self, state):
        obs = state['obs']
        legal_actions = list(state['legal_actions'].keys())
        self.current_hand_sequence.append(obs)
        if np.random.rand() < self.epsilon:
            return np.random.choice(legal_actions)
        return self._greedy_action(legal_actions)

    def eval_step(self, state):
        obs = state['obs']
        legal_actions = list(state['legal_actions'].keys())
        self.current_hand_sequence.append(obs)
        return self._greedy_action(legal_actions), {}

    def feed(self, transition):
        state, action, reward, next_state, done = transition
        if done and len(self.current_hand_sequence) > 0:
            self.replay_buffer.push(self.current_hand_sequence, action, reward)
            self.current_hand_sequence = []

    def train(self):
        if len(self.replay_buffer) < self.min_replay:
            return None

        batch = self.replay_buffer.sample(self.batch_size)
        max_len = max(len(seq) for seq, _, _ in batch)
        feat = batch[0][0][0].shape[0]

        padded = np.zeros((self.batch_size, max_len, feat), dtype=np.float32)
        actions = np.zeros(self.batch_size, dtype=np.int64)
        rewards = np.zeros(self.batch_size, dtype=np.float32)

        for i, (seq, action, reward) in enumerate(batch):
            seq_arr = np.array(seq, dtype=np.float32)
            padded[i, :len(seq)] = seq_arr
            actions[i] = action
            rewards[i] = reward

        states_t = torch.from_numpy(padded).to(self.device)
        actions_t = torch.from_numpy(actions).long().unsqueeze(1).to(self.device)
        rewards_t = torch.from_numpy(rewards).to(self.device)

        self.model.train()
        current_q, _ = self.model(states_t)
        current_q = current_q.gather(1, actions_t).squeeze(1)
        target_q = rewards_t

        loss = F.smooth_l1_loss(current_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

        self._train_steps += 1
        if self._train_steps % self.target_update_freq == 0:
            self.target_model.load_state_dict(self.model.state_dict())

        return loss.item()

    def save_model(self, path):
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'target_model_state_dict': self.target_model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'epsilon': self.epsilon,
            'train_steps': self._train_steps
        }, path)


class ConservativeHeuristicAgent:
    """Rule-based agent betting on card strength."""
    
    def __init__(self, num_actions):
        self.use_raw = True
        self.num_actions = num_actions

    def step(self, state):
        action, _ = self.eval_step(state)
        return action

    def eval_step(self, state):
        raw_obs = state['raw_obs']
        legal_actions = raw_obs['legal_actions']
        raw_hand = raw_obs['hand']
        hand_str = raw_hand[0] if isinstance(raw_hand, list) else raw_hand
        rank = hand_str[-1]
        raw_public = raw_obs['public_card']
        public_rank = None
        if raw_public:
            pub_str = raw_public[0] if isinstance(raw_public, list) else raw_public
            public_rank = pub_str[-1]

        if public_rank:
            if public_rank == rank:
                if 'raise' in legal_actions: return 'raise', {}
                if 'call' in legal_actions: return 'call', {}
            if rank == 'K':
                if 'raise' in legal_actions: return 'raise', {}
                if 'call' in legal_actions: return 'call', {}
            if rank == 'Q':
                if 'check' not in legal_actions:
                    if 'fold' in legal_actions: return 'fold', {}
                if 'check' in legal_actions: return 'check', {}
                if 'call' in legal_actions: return 'call', {}
            if rank == 'J':
                if 'check' in legal_actions: return 'check', {}
                if 'fold' in legal_actions: return 'fold', {}
        else:
            if rank == 'K':
                if 'raise' in legal_actions: return 'raise', {}
                if 'call' in legal_actions: return 'call', {}
            if rank == 'Q':
                if 'check' in legal_actions: return 'check', {}
                if 'call' in legal_actions: return 'call', {}
            if rank == 'J':
                if 'check' in legal_actions: return 'check', {}
                if 'fold' in legal_actions: return 'fold', {}

        return legal_actions[0], {}

print("✓ Agents defined")

In [ ]:
# ============================================================
# TRAINING (inline version)
# ============================================================

import rlcard
from rlcard.utils import reorganize

class CurriculumTrainer:
    """Two-phase curriculum training."""
    
    def __init__(self, env, agent, opponent_phase1, opponent_phase2):
        self.env = env
        self.agent = agent
        self.opponent_phase1 = opponent_phase1
        self.opponent_phase2 = opponent_phase2
        self.eval_random = opponent_phase1
        self.ev_history_random = []
        self.ev_history_heuristic = []
        self.loss_history = []

    def train_episode(self):
        trajectories, payoffs = self.env.run(is_training=True)
        trajectories = reorganize(trajectories, payoffs)
        for ts in trajectories[0]:
            self.agent.feed(ts)
        loss = self.agent.train()
        return loss

    def evaluate(self, opponent, num_hands=500):
        self.env.set_agents([self.agent, opponent])
        total = 0
        for _ in range(num_hands):
            if hasattr(self.agent, 'current_hand_sequence'):
                self.agent.current_hand_sequence = []
            _, payoffs = self.env.run(is_training=False)
            total += payoffs[0]
        return total / num_hands

    def train_phase1(self, num_episodes, eval_every, eval_num):
        print("=" * 60)
        print(f"PHASE 1: Training vs Random Agent (0-{num_episodes})")
        print("=" * 60)
        self.env.set_agents([self.agent, self.opponent_phase1])
        
        for episode in range(num_episodes):
            loss = self.train_episode()
            if loss is not None:
                self.loss_history.append((episode, loss))
            
            if episode % eval_every == 0:
                ev_random = self.evaluate(self.eval_random, eval_num)
                ev_heuristic = self.evaluate(self.opponent_phase2, eval_num)
                self.ev_history_random.append((episode, ev_random))
                self.ev_history_heuristic.append((episode, ev_heuristic))
                epsilon = getattr(self.agent, 'epsilon', 0.0)
                print(f"[P1] Ep {episode:05d} | ε={epsilon:.3f} | "
                      f"EV vs Random: {ev_random:+.3f} | "
                      f"EV vs Heuristic: {ev_heuristic:+.3f}")
                self.env.set_agents([self.agent, self.opponent_phase1])

    def train_phase2(self, start_episode, num_episodes, eval_every, eval_num):
        print()
        print("=" * 60)
        print(f"PHASE 2: Fine-tuning vs Heuristic ({start_episode}-{start_episode + num_episodes})")
        print("=" * 60)
        self.env.set_agents([self.agent, self.opponent_phase2])
        
        for episode in range(start_episode, start_episode + num_episodes):
            loss = self.train_episode()
            if loss is not None:
                self.loss_history.append((episode, loss))
            
            if episode % eval_every == 0:
                ev_random = self.evaluate(self.eval_random, eval_num)
                ev_heuristic = self.evaluate(self.opponent_phase2, eval_num)
                self.ev_history_random.append((episode, ev_random))
                self.ev_history_heuristic.append((episode, ev_heuristic))
                epsilon = getattr(self.agent, 'epsilon', 0.0)
                print(f"[P2] Ep {episode:05d} | ε={epsilon:.3f} | "
                      f"EV vs Random: {ev_random:+.3f} | "
                      f"EV vs Heuristic: {ev_heuristic:+.3f}")
                self.env.set_agents([self.agent, self.opponent_phase2])

    def train(self):
        self.train_phase1(NUM_EPISODES_PHASE1, EVALUATE_EVERY, EVALUATE_NUM)
        self.train_phase2(NUM_EPISODES_PHASE1, NUM_EPISODES_PHASE2,
                         EVALUATE_EVERY, EVALUATE_NUM)
        print()
        print("--- TRAINING COMPLETE ---")
        return {
            'ev_random': self.ev_history_random,
            'ev_heuristic': self.ev_history_heuristic,
            'loss': self.loss_history
        }

print("✓ Training module defined")

In [ ]:
# ============================================================
# EVALUATION (inline version)
# ============================================================

def evaluate_agents(env, agent_a, agent_b, num_hands=1000, label=''):
    """Evaluate agent_a against agent_b."""
    env.set_agents([agent_a, agent_b])
    total = 0
    trajectories_collected = []
    
    for _ in range(num_hands):
        if hasattr(agent_a, 'current_hand_sequence'):
            agent_a.current_hand_sequence = []
        trajectories, payoffs = env.run(is_training=False)
        total += payoffs[0]
        trajectories_collected.append(trajectories[0])
    
    avg_ev = total / num_hands
    if label:
        print(f'  {label:30s}: {avg_ev:+.3f} chips/hand')
    return avg_ev, trajectories_collected


def compute_action_distribution(trajectories, action_names):
    """Compute action distribution and bluff rate."""
    action_counts = {name: 0 for name in action_names.values()}
    bluff_attempts = 0
    total_jack_steps = 0
    
    for hand_trajectory in trajectories:
        for state in hand_trajectory:
            if not isinstance(state, dict):
                continue
            raw_obs = state.get('raw_obs', {})
            action_record = state.get('action_record', [])
            player0_actions = [a for pid, a in action_record if pid == 0]
            if not player0_actions:
                continue
            
            action_str = player0_actions[-1]
            name = action_names.get(action_str, 'Unknown')
            action_counts[name] = action_counts.get(name, 0) + 1
            
            raw_hand = raw_obs.get('hand', '')
            rank = raw_hand[-1] if isinstance(raw_hand, str) and raw_hand else ''
            if rank == 'J':
                total_jack_steps += 1
                if action_str == 'raise':
                    bluff_attempts += 1
    
    total_actions = sum(action_counts.values())
    bluff_rate = 100 * bluff_attempts / total_jack_steps if total_jack_steps else 0
    
    return {
        'action_counts': action_counts,
        'total_actions': total_actions,
        'bluff_attempts': bluff_attempts,
        'total_jack_steps': total_jack_steps,
        'bluff_rate': bluff_rate
    }

print("✓ Evaluation functions defined")

In [ ]:
# ============================================================
# VISUALIZATION (inline version)
# ============================================================

import matplotlib.pyplot as plt

def plot_training_curves(ev_history_random, ev_history_heuristic, loss_history,
                        phase1_episodes, save_path=None):
    """Plot EV and loss curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('DRQN Two-Phase Curriculum Training',
                 fontsize=14, fontweight='bold')
    
    if ev_history_random:
        eps_r, evs_r = zip(*ev_history_random)
        axes[0].plot(eps_r, evs_r, 'b-o', markersize=4, label='vs Random')
    
    if ev_history_heuristic:
        eps_h, evs_h = zip(*ev_history_heuristic)
        axes[0].plot(eps_h, evs_h, 'g-s', markersize=4, label='vs Heuristic')
    
    axes[0].axhline(0, color='gray', linestyle='--', linewidth=0.8)
    axes[0].axvline(phase1_episodes, color='orange', linestyle=':',
                   linewidth=1.5, label='Phase 2 starts')
    axes[0].set_xlabel('Episode')
    axes[0].set_ylabel('Avg chips / hand')
    axes[0].set_title('Expected Value — Both Opponents')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    if loss_history:
        eps_l, losses = zip(*loss_history)
        window = min(200, len(losses) // 10)
        if window > 1:
            smoothed = np.convolve(losses, np.ones(window) / window, mode='valid')
            axes[1].plot(eps_l[window-1:], smoothed, 'r-',
                        linewidth=1.5, label='Smoothed')
        axes[1].plot(eps_l, losses, 'r-', alpha=0.12,
                    linewidth=0.5, label='Raw')
        axes[1].axvline(phase1_episodes, color='orange', linestyle=':',
                       linewidth=1.5, label='Phase 2 starts')
        axes[1].set_xlabel('Episode')
        axes[1].set_ylabel('Huber Loss')
        axes[1].set_title('Training Loss')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved training curves to {save_path}')
    plt.show()


def plot_ev_comparison_bar(results, save_path=None):
    """Plot bar chart of EV comparison."""
    labels = list(results.keys())
    values = list(results.values())
    colors = ['steelblue' if v >= 0 else 'tomato' for v in values]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=1.2)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Average chips / hand')
    ax.set_title('Final EV Comparison — DRQN vs All Baselines',
                fontsize=13, fontweight='bold')
    
    for bar, val in zip(bars, values):
        offset = 0.03 if val >= 0 else -0.07
        ax.text(bar.get_x() + bar.get_width() / 2, val + offset,
                f'{val:+.3f}', ha='center', va='bottom',
                fontsize=10, fontweight='bold')
    
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved EV comparison to {save_path}')
    plt.show()

print("✓ Visualization functions defined")

## 🚀 Main Training Pipeline

### 1. Environment Setup

In [ ]:
from rlcard.utils import set_seed
from rlcard.agents import RandomAgent, DQNAgent

set_seed(SEED)
env = rlcard.make(ENV_NAME)

raw_shape = env.state_shape[0]
state_shape = raw_shape[0] if isinstance(raw_shape, list) else raw_shape
num_actions = env.num_actions

print(f'Environment: {ENV_NAME}')
print(f'State shape: {state_shape} | Num actions: {num_actions}')
print(f'Device: {DEVICE}')

### 2. Initialize Agents

In [ ]:
# DRQN Agent (with LSTM memory)
agent_drqn = DRQNAgent(
    state_shape=state_shape,
    num_actions=num_actions,
    device=DEVICE,
    hidden_size=HIDDEN_SIZE,
    lr=LEARNING_RATE,
    gamma=GAMMA,
    epsilon_start=EPSILON_START,
    epsilon_min=EPSILON_MIN,
    epsilon_decay=EPSILON_DECAY,
    buffer_capacity=BUFFER_CAPACITY,
    batch_size=BATCH_SIZE,
    min_replay=MIN_REPLAY_SIZE,
    target_update_freq=TARGET_UPDATE_FREQ
)

# DQN Agent (feedforward baseline)
agent_dqn = DQNAgent(
    num_actions=num_actions,
    state_shape=env.state_shape[0],
    mlp_layers=[HIDDEN_SIZE, HIDDEN_SIZE],
    device=DEVICE
)

# Baseline agents
agent_random = RandomAgent(num_actions)
agent_heuristic = ConservativeHeuristicAgent(num_actions)

print("✓ DRQN Agent initialized")
print("✓ DQN Agent initialized")
print("✓ Random Agent initialized")
print("✓ Heuristic Agent initialized")

### 3. Train DRQN (Two-Phase Curriculum)

**Note**: Training is reduced to 5K episodes per phase for Colab. Increase in `config` for better performance.

In [ ]:
trainer_drqn = CurriculumTrainer(
    env=env,
    agent=agent_drqn,
    opponent_phase1=agent_random,
    opponent_phase2=agent_heuristic
)

drqn_history = trainer_drqn.train()

# Save model
drqn_checkpoint = f'{MODEL_DIR}/drqn_final.pt'
agent_drqn.save_model(drqn_checkpoint)
print(f"\n✓ DRQN training complete. Model saved to {drqn_checkpoint}")

### 4. Train DQN (Baseline Comparison)

In [ ]:
trainer_dqn = CurriculumTrainer(
    env=env,
    agent=agent_dqn,
    opponent_phase1=agent_random,
    opponent_phase2=agent_heuristic
)

dqn_history = trainer_dqn.train()
print("\n✓ DQN training complete")

### 5. Training Curves Visualization

In [ ]:
plot_training_curves(
    ev_history_random=drqn_history['ev_random'],
    ev_history_heuristic=drqn_history['ev_heuristic'],
    loss_history=drqn_history['loss'],
    phase1_episodes=NUM_EPISODES_PHASE1,
    save_path=f'{PLOT_DIR}/drqn_training_curves.png'
)

### 6. Final Evaluation

In [ ]:
print(f"\n{'='*60}")
print(f"FINAL EVALUATION ({NUM_EVAL_HANDS} hands per matchup)")
print(f"{'='*60}")

results = {}

# DRQN evaluations
print('\nDRQN vs ...')
ev_vs_random, traj_drqn_vs_random = evaluate_agents(
    env, agent_drqn, agent_random, NUM_EVAL_HANDS, 'Random'
)
ev_vs_heuristic, traj_drqn_vs_heuristic = evaluate_agents(
    env, agent_drqn, agent_heuristic, NUM_EVAL_HANDS, 'Heuristic'
)
ev_vs_dqn, traj_drqn_vs_dqn = evaluate_agents(
    env, agent_drqn, agent_dqn, NUM_EVAL_HANDS, 'Standard DQN'
)

# DQN evaluations
print('\nDQN vs ...')
dqn_vs_random, traj_dqn_vs_random = evaluate_agents(
    env, agent_dqn, agent_random, NUM_EVAL_HANDS, 'Random'
)
dqn_vs_heuristic, traj_dqn_vs_heuristic = evaluate_agents(
    env, agent_dqn, agent_heuristic, NUM_EVAL_HANDS, 'Heuristic'
)

# Store results
results['DRQN vs Random'] = {
    'ev': ev_vs_random,
    'action_stats': compute_action_distribution(traj_drqn_vs_random, ACTION_NAMES)
}
results['DRQN vs Heuristic'] = {
    'ev': ev_vs_heuristic,
    'action_stats': compute_action_distribution(traj_drqn_vs_heuristic, ACTION_NAMES)
}
results['DRQN vs DQN'] = {
    'ev': ev_vs_dqn,
    'action_stats': compute_action_distribution(traj_drqn_vs_dqn, ACTION_NAMES)
}
results['DQN vs Random'] = {
    'ev': dqn_vs_random
}
results['DQN vs Heuristic'] = {
    'ev': dqn_vs_heuristic
}

### 7. Results Analysis

In [ ]:
# Detailed analysis
print("\n" + "="*60)
print("DETAILED ANALYSIS")
print("="*60)

for matchup, metrics in results.items():
    if 'action_stats' in metrics:
        print(f"\n{matchup}:")
        print(f"  EV: {metrics['ev']:+.3f} chips/hand")
        stats = metrics['action_stats']
        print(f"  Bluff Rate: {stats['bluff_rate']:.1f}% "
              f"({stats['bluff_attempts']}/{stats['total_jack_steps']} Jack decisions)")

# Final verdict
print("\n" + "="*60)
if ev_vs_dqn > 0:
    print("✓ SUCCESS: DRQN outperforms the memoryless DQN!")
    print(f"  DRQN advantage: {ev_vs_dqn:+.3f} chips/hand")
else:
    print("⚠ NOTE: DQN held its ground — consider more training episodes.")
print("="*60)

In [ ]:
# EV comparison bar chart
ev_comparison = {
    'DRQN\nvs Random': results['DRQN vs Random']['ev'],
    'DRQN\nvs Heuristic': results['DRQN vs Heuristic']['ev'],
    'DRQN\nvs DQN': results['DRQN vs DQN']['ev'],
    'DQN\nvs Random': results['DQN vs Random']['ev'],
    'DQN\nvs Heuristic': results['DQN vs Heuristic']['ev'],
}
plot_ev_comparison_bar(
    results=ev_comparison,
    save_path=f'{PLOT_DIR}/ev_comparison.png'
)

### 8. Download Results (Optional)

Download trained models and plots to your local machine:

In [ ]:
# Zip outputs for easy download
!zip -r /content/dlpw_outputs.zip {OUTPUT_DIR}

# Download using Colab's file browser or:
from google.colab import files
files.download('/content/dlpw_outputs.zip')

print("✓ Results zipped and ready for download")

### 9. Save to Google Drive (Optional)

In [ ]:
# Uncomment to save outputs to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# 
# !cp -r {OUTPUT_DIR} '/content/drive/MyDrive/DLPW_outputs'
# print("✓ Outputs saved to Google Drive")

## 📊 Key Findings

### Success Criterion
**DRQN must outperform DQN** to prove that recurrent memory is beneficial for imperfect information games.

### Expected Results
- **DRQN vs DQN**: Positive EV (memory advantage)
- **Bluff Rate**: 15-20% raising with Jack (proves strategic reasoning)
- **Phase 2 Improvement**: Better performance vs Heuristic after curriculum learning

### Interpretation
- LSTM enables the agent to track betting sequences
- This allows inference of opponent tendencies
- Bluffing behavior emerges without explicit programming

---

## 🚀 Next Steps

1. **Extended Training**: Increase episodes in config (15K+ per phase)
2. **Hyperparameter Tuning**: Adjust learning rate, epsilon decay
3. **Self-Play**: Train DRQN against itself
4. **Larger Games**: Scale to Texas Hold'em

See full project at: `https://github.com/YOUR_USERNAME/DLPW`